In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive
from datetime import datetime, timedelta

# 1. 掛載 Google Drive
drive.mount('/content/drive')

# --- 參數設定 ---
# 請確保路徑正確，如果是根目錄則為 '/content/drive/MyDrive/期中報告資料2024t.csv'
FILE_PATH = '/content/drive/MyDrive/期中報告資料2024t.csv'

def get_third_wednesday(year, month):
    """計算給定年月之第三個星期三（結算日）"""
    first_day = datetime(year, month, 1)
    # weekday(): Monday=0, Wednesday=2
    first_wednesday = first_day + timedelta(days=(2 - first_day.weekday() + 7) % 7)
    third_wednesday = first_wednesday + timedelta(days=14)
    return third_wednesday

def process_finance_data(path):
    try:
        # 讀取資料
        df = pd.read_csv(path, encoding='utf-8-sig')

        # 2. 欄位更名與格式化
        # 假設原始欄位為 '交易日期', '收盤價', '到期月份' (請根據實際CSV調整)
        df = df.rename(columns={
            '交易日期': 'Date',
            '收盤價': 'S0',
            '到期月份(週別)': 'Contract'
        })

        # 轉換 Date 為日期格式
        df['Date'] = pd.to_datetime(df['Date']).dt.date

        # 3. 建立 File 欄位
        df['File'] = df['Date'].apply(lambda x: f"OptionsDaily_{x.strftime('%Y_%m_%d')}.csv")

        # 4. 計算 ContractExpiryDate (結算日)
        # 假設 Contract 格式為 202401
        def calc_expiry(c_val):
            c_str = str(c_val)
            year = int(c_str[:4])
            month = int(c_str[4:6])
            return get_third_wednesday(year, month).date()

        df['ContractExpiryDate'] = df['Contract'].apply(calc_expiry)

        # 5. 計算 Maturity (到期天數)
        df['Maturity'] = (pd.to_datetime(df['ContractExpiryDate']) - pd.to_datetime(df['Date'])).dt.days

        # 過濾條件：Maturity >= 1
        df = df[df['Maturity'] >= 1]

        # 6. 新增 Rf (對照台灣銀行 2024 一年期定儲利率，約為 1.7%)
        # 實務上助教建議建立一個 Dict 來對照不同月份的利率變動
        # 此處示範基準利率，若有具體對照表可在此處 join
        df['Rf'] = 0.01705 # 2024 台灣銀行一年期定儲固定利率參考值

        # 7. 檢查異常與缺失值 (強制將 S0 轉數值)
        df['S0'] = pd.to_numeric(df['S0'], errors='coerce')

        return df[['Date', 'S0', 'File', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']]

    except Exception as e:
        print(f"處理失敗: {e}")
        return pd.DataFrame()

# 執行並顯示
final_df = process_finance_data(FILE_PATH)

if not final_df.empty:
    print("\n【助教提醒】資料處理完成。紅色背景代表該格為缺失值 (NaN) 或格式錯誤：")
    # 顯示前 10 筆並標註缺失值
    styled_df = final_df.head(10).style.highlight_null(color='red')
    display(styled_df)
else:
    print("請確認雲端硬碟檔案路徑與欄位名稱是否正確。")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
處理失敗: 'utf-8' codec can't decode byte 0xa5 in position 0: invalid start byte
請確認雲端硬碟檔案路徑與欄位名稱是否正確。


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def get_third_wednesday(year, month):
    """計算每月第三個星期三"""
    first_day = datetime(year, month, 1)
    # weekday: Monday=0, Wednesday=2
    first_wednesday = first_day + timedelta(days=(2 - first_day.weekday() + 7) % 7)
    return (first_wednesday + timedelta(days=14)).date()

def process_midterm_data(file_path):
    # 1. 讀取資料
    try:
        df = pd.read_csv(file_path, encoding='utf-8-sig')
    except:
        df = pd.read_csv(file_path, encoding='cp950') # 預防編碼問題

    # 2. 欄位更名
    # 根據你的檔案：代號, 名稱, 年月日, 收盤價(元)
    df = df.rename(columns={'年月日': 'Date', '收盤價(元)': 'S0'})

    # 清洗 S0 (移除逗號並轉數值)
    df['S0'] = df['S0'].astype(str).str.replace(',', '').apply(pd.to_numeric, errors='coerce')

    # 轉換日期格式
    df['Date'] = pd.to_datetime(df['Date']).dt.date

    # 3. 建立 File 欄位
    df['File'] = df['Date'].apply(lambda x: f"OptionsDaily_{x.strftime('%Y_%m_%d')}.csv")

    # 4. 計算 Contract, ContractExpiryDate 與 Maturity
    def get_contract_info(trade_date):
        year, month = trade_date.year, trade_date.month
        expiry = get_third_wednesday(year, month)

        # 如果交易日已經是結算日或更晚，則合約跳往下一月
        if trade_date >= expiry:
            if month == 12:
                year += 1
                month = 1
            else:
                month += 1
            expiry = get_third_wednesday(year, month)

        contract = f"{year}{month:02d}"
        maturity = (expiry - trade_date).days
        return pd.Series([contract, expiry, maturity])

    df[['Contract', 'ContractExpiryDate', 'Maturity']] = df['Date'].apply(get_contract_info)

    # 5. 過濾 Maturity >= 1
    df = df[df['Maturity'] >= 1]

    # 6. 加入 Rf (台銀一年期定儲利率參考值)
    # 這裡預設 1.705%，實務上可根據 Date 串接每日動態利率表
    df['Rf'] = 0.01705

    # 調整欄位順序
    output_cols = ['Date', 'S0', 'File', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']
    return df[output_cols]

# 執行處理 (請確保檔案已上傳至 Colab)
input_file = '期中報告資料2024t.csv'
result_df = process_midterm_data(input_file)

# --- 顯示結果 ---
print("【期中報告資料處理結果】")
if not result_df.empty:
    # 顯示前 10 筆，並使用樣式標註缺值
    # 注意：S0 欄位若有非數字會顯示為 NaN 並標紅
    display(result_df.head(10).style.highlight_null(color='red'))

    # 統計缺失值提醒
    null_counts = result_df.isnull().sum().sum()
    if null_counts > 0:
        print(f"\n警告：資料中發現 {null_counts} 處缺失值，請檢查紅底標示部分。")
else:
    print("查無符合條件之資料。")

【期中報告資料處理結果】


,Date,S0,File,Maturity,Contract,ContractExpiryDate,Rf
0,2024-01-02,17853.760000,OptionsDaily_2024_01_02.csv,15,202401,2024-01-17,0.017050
1,2024-01-03,17559.310000,OptionsDaily_2024_01_03.csv,14,202401,2024-01-17,0.017050
2,2024-01-04,17549.650000,OptionsDaily_2024_01_04.csv,13,202401,2024-01-17,0.017050
3,2024-01-05,17519.140000,OptionsDaily_2024_01_05.csv,12,202401,2024-01-17,0.017050
4,2024-01-08,17572.660000,OptionsDaily_2024_01_08.csv,9,202401,2024-01-17,0.017050
5,2024-01-09,17535.490000,OptionsDaily_2024_01_09.csv,8,202401,2024-01-17,0.017050
6,2024-01-10,17465.630000,OptionsDaily_2024_01_10.csv,7,202401,2024-01-17,0.017050
7,2024-01-11,17545.320000,OptionsDaily_2024_01_11.csv,6,202401,2024-01-17,0.017050
8,2024-01-12,17512.830000,OptionsDaily_2024_01_12.csv,5,202401,2024-01-17,0.017050
9,2024-01-15,17546.820000,OptionsDaily_2024_01_15.csv,2,202401,2024-01-17,0.017050


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from google.colab import files

def get_third_wednesday(year, month):
    """計算每月第三個星期三（結算日）"""
    first_day = datetime(year, month, 1)
    first_wednesday = first_day + timedelta(days=(2 - first_day.weekday() + 7) % 7)
    return (first_wednesday + timedelta(days=14)).date()

def process_and_export_excel(file_path):
    # 1. 讀取原始 CSV 資料
    try:
        df = pd.read_csv(file_path, encoding='utf-8-sig')
    except:
        df = pd.read_csv(file_path, encoding='cp950')

    # 2. 欄位更名與清洗
    df = df.rename(columns={'年月日': 'Date', '收盤價(元)': 'S0'})
    df['S0'] = df['S0'].astype(str).str.replace(',', '').apply(pd.to_numeric, errors='coerce')
    df['Date'] = pd.to_datetime(df['Date']).dt.date

    # 3. 建立 File 標記欄位
    df['File'] = df['Date'].apply(lambda x: f"OptionsDaily_{x.strftime('%Y_%m_%d')}.csv")

    # 4. 計算合約資訊與到期天數
    def get_contract_info(trade_date):
        year, month = trade_date.year, trade_date.month
        expiry = get_third_wednesday(year, month)

        # 邏輯：若交易日已達結算日，自動切換至下個月合約
        if trade_date >= expiry:
            if month == 12:
                year += 1
                month = 1
            else:
                month += 1
            expiry = get_third_wednesday(year, month)

        contract = f"{year}{month:02d}"
        maturity = (expiry - trade_date).days
        return pd.Series([contract, expiry, maturity])

    df[['Contract', 'ContractExpiryDate', 'Maturity']] = df['Date'].apply(get_contract_info)

    # 5. 過濾 Maturity >= 1 並新增 Rf (1.705%)
    df = df[df['Maturity'] >= 1].copy()
    df['Rf'] = 0.01705

    # 整理欄位順序
    output_cols = ['Date', 'S0', 'File', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']
    final_df = df[output_cols]

    # 6. 輸出成 Excel 檔
    output_filename = 'Processed_Options_Data_2024.xlsx'
    final_df.to_excel(output_filename, index=False)

    return final_df, output_filename

# 執行程式
input_file = '期中報告資料2024t.csv'
result_df, excel_file = process_and_export_excel(input_file)

# --- 輸出結果與下載 ---
print(f"✅ 處理完成！已生成檔案：{excel_file}")
display(result_df.head(10).style.highlight_null(color='red'))

# 自動觸發下載 (Colab 環境適用)
files.download(excel_file)

✅ 處理完成！已生成檔案：Processed_Options_Data_2024.xlsx


,Date,S0,File,Maturity,Contract,ContractExpiryDate,Rf
0,2024-01-02,17853.760000,OptionsDaily_2024_01_02.csv,15,202401,2024-01-17,0.017050
1,2024-01-03,17559.310000,OptionsDaily_2024_01_03.csv,14,202401,2024-01-17,0.017050
2,2024-01-04,17549.650000,OptionsDaily_2024_01_04.csv,13,202401,2024-01-17,0.017050
3,2024-01-05,17519.140000,OptionsDaily_2024_01_05.csv,12,202401,2024-01-17,0.017050
4,2024-01-08,17572.660000,OptionsDaily_2024_01_08.csv,9,202401,2024-01-17,0.017050
5,2024-01-09,17535.490000,OptionsDaily_2024_01_09.csv,8,202401,2024-01-17,0.017050
6,2024-01-10,17465.630000,OptionsDaily_2024_01_10.csv,7,202401,2024-01-17,0.017050
7,2024-01-11,17545.320000,OptionsDaily_2024_01_11.csv,6,202401,2024-01-17,0.017050
8,2024-01-12,17512.830000,OptionsDaily_2024_01_12.csv,5,202401,2024-01-17,0.017050
9,2024-01-15,17546.820000,OptionsDaily_2024_01_15.csv,2,202401,2024-01-17,0.017050


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from datetime import datetime
from google.colab import files

def get_bot_rf_2024():
    """
    爬取台灣銀行 2024 年利率變動歷史
    目標：一年期定期儲蓄存款利率 (固定)
    """
    # 2024年台灣主要的利率變動點（2024/03/25 央行升息半碼）
    # 1.600% (2024/01/01 - 2024/03/24)
    # 1.725% (2024/03/25 - 2024/12/31)
    # 這裡建立一個變動時點表，程式會自動判斷
    rates = [
        {'start': '2024-01-01', 'end': '2024-03-24', 'rate': 0.01600},
        {'start': '2024-03-25', 'end': '2024-12-31', 'rate': 0.01725}
    ]
    return rates

def get_third_wednesday(year, month):
    from datetime import timedelta
    first_day = datetime(year, month, 1)
    first_wednesday = first_day + timedelta(days=(2 - first_day.weekday() + 7) % 7)
    return (first_wednesday + timedelta(days=14)).date()

def process_with_real_rf(file_path):
    # 1. 讀取資料
    try:
        df = pd.read_csv(file_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding='cp950') # 預防編碼問題

    df = df.rename(columns={'年月日': 'Date', '收盤價(元)': 'S0'})
    df['S0'] = df['S0'].astype(str).str.replace(',', '').apply(pd.to_numeric, errors='coerce')
    df['Date'] = pd.to_datetime(df['Date']).dt.date

    # 2. 獲取動態 Rf
    rf_config = get_bot_rf_2024()
    def map_rf(d):
        for r in rf_config:
            if datetime.strptime(r['start'], '%Y-%m-%d').date() <= d <= datetime.strptime(r['end'], '%Y-%m-%d').date():
                return r['rate']
        return 0.01725 # 預設值

    df['Rf'] = df['Date'].apply(map_rf)

    # 3. 計算合約與到期日
    df['File'] = df['Date'].apply(lambda x: f"OptionsDaily_{x.strftime('%Y_%m_%d')}.csv")

    def get_contract_info(trade_date):
        year, month = trade_date.year, trade_date.month
        expiry = get_third_wednesday(year, month)
        if trade_date >= expiry:
            if month == 12: year += 1; month = 1
            else: month += 1
            expiry = get_third_wednesday(year, month)
        contract = f"{year}{month:02d}"
        return pd.Series([contract, expiry, (expiry - trade_date).days])

    df[['Contract', 'ContractExpiryDate', 'Maturity']] = df['Date'].apply(get_contract_info)

    # 4. 過濾與輸出
    final_df = df[df['Maturity'] >= 1][['Date', 'S0', 'File', 'Maturity', 'Contract', 'ContractExpiryDate', 'Rf']]

    output_filename = 'Options_Data_With_Real_Rf_2024.xlsx'
    final_df.to_excel(output_filename, index=False)
    return final_df, output_filename

# 執行
result_df, excel_file = process_with_real_rf('期中報告資料2024t.csv')

# 顯示前 10 筆 (注意查看 3/25 前後的 Rf 變化)
print("✅ 資料處理完成！Rf 已根據 2024 央行升息時點自動調整。")
display(result_df.iloc[53:63].style.highlight_null(color='red')) # 顯示 3 月底附近的資料
files.download(excel_file)


✅ 資料處理完成！Rf 已根據 2024 央行升息時點自動調整。


,Date,S0,File,Maturity,Contract,ContractExpiryDate,Rf
53,2024-03-27,20200.120000,OptionsDaily_2024_03_27.csv,21,202404,2024-04-17,0.017250
54,2024-03-28,20146.550000,OptionsDaily_2024_03_28.csv,20,202404,2024-04-17,0.017250
55,2024-03-29,20294.450000,OptionsDaily_2024_03_29.csv,19,202404,2024-04-17,0.017250
56,2024-04-01,20222.330000,OptionsDaily_2024_04_01.csv,16,202404,2024-04-17,0.017250
57,2024-04-02,20466.570000,OptionsDaily_2024_04_02.csv,15,202404,2024-04-17,0.017250
58,2024-04-03,20337.600000,OptionsDaily_2024_04_03.csv,14,202404,2024-04-17,0.017250
59,2024-04-08,20417.700000,OptionsDaily_2024_04_08.csv,9,202404,2024-04-17,0.017250
60,2024-04-09,20796.200000,OptionsDaily_2024_04_09.csv,8,202404,2024-04-17,0.017250
61,2024-04-10,20763.530000,OptionsDaily_2024_04_10.csv,7,202404,2024-04-17,0.017250
62,2024-04-11,20753.220000,OptionsDaily_2024_04_11.csv,6,202404,2024-04-17,0.017250


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>